# So sánh bảy OCR FP32 sau train cùng dataset

Kernel **VN-LPR CPU benchmark**. Đo trên **i5-14600KF**, không ghép tốc độ ThinkPad. VietLPR_v1 là model đã chốt trước đây tên VietLPR_v2; model VietLPR_v1 cũ không tham gia. Không dùng bảng 14 pretrained trước đây làm kết quả mới.

In [ ]:
from pathlib import Path
import os, sys
ROOT = Path.cwd().resolve()
if not (ROOT / "configs/ocr/adaptation.json").is_file():
    ROOT = ROOT.parent
assert (ROOT / "configs/ocr/adaptation.json").is_file(), "Mở notebook từ dự án"
os.chdir(ROOT)
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from ocr.adaptation.notebook import run
from ocr.adaptation.common import settings, directory
print(ROOT)


## 1. Kiểm bảy model
Chỉ test khi đã chọn best và kiểm export trên val; không dùng GT box hỗ trợ inference.

In [ ]:
from ocr.adaptation.common import validate_frozen
for model in settings()["models"]:
    try:
        state = validate_frozen(model["name"])
        print(model["name"], state["precision"])
    except FileNotFoundError:
        print(model["name"], "chưa train/export xong")

## 2. Benchmark chung
Năm vòng × bảy model = **35 pass**. Mỗi pass mở tiến trình mới, cố định bốn lõi P, bốn luồng intra-op/một inter-op, tắt spin wait; warm-up 20. Đọc ảnh trước vào RAM; không I/O trong vùng đo. Speed gồm toàn pipeline OCR đến chuỗi chữ. ACC/CER trên đủ 124 ảnh test.

In [ ]:
RUN_BENCHMARK = False
RESUME_RUN = None  # Dán run_dir trong runs/ocr_adaptation/active_benchmark.json nếu bị ngắt
if RUN_BENCHMARK:
    command = ["-m", "ocr.adaptation.benchmark", "run"]
    if RESUME_RUN: command += ["--run-dir", str(RESUME_RUN)]
    run(ROOT / ".venv-benchmark/Scripts/python.exe", *command)

## 3. Bảng kết quả
Rank theo ACC giảm → CER tăng → Speed tăng; Speedup lấy model chậm nhất làm 1×. SD là độ lệch chuẩn mẫu của năm trung bình vòng, không phải năm lần train.

In [ ]:
from IPython.display import Markdown, display
from ocr.adaptation.benchmark import table
display(Markdown(table()))